# LF reversal — LINCS L1000 dry-run (pilot signature)

Queries **L1000CDS2** (Ma'ayan Lab) in **reverse mode** for perturbagens whose L1000 transcriptional signature *opposes* the LF activated-fibroblast signature from the GSE294458 pilot, then applies the program's guards (novelty vs LF prior art, cytotoxicity caution).

> **DRY-RUN — NON-CONFIRMATORY.** Built on a 1-vs-1 pilot signature. L1000CDS2 is known to return HDAC / proteasome / topoisomerase inhibitors for almost any query (they globally shut down transcription and thus "reverse" many signatures), so a strong reversal score is necessary but nowhere near sufficient. Nothing here is a candidate until (a) the signature is rebuilt on the replicated Korea U / SMU cohorts and (b) survivors pass the viability / matrix-preservation and local-delivery guards. Today's purpose: exercise the pipeline and preview candidate classes.

In [ ]:
# 1. Imports
import os, json, re
import requests
import pandas as pd
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("ready; colab =", IN_COLAB)

## 1 — Load the pilot signature

Loads `out/LF_reversal_signature.json` written by the pilot notebook. If it is not present in this runtime, you will be prompted to upload it. A second, tighter noise scrub is applied here (adds the `EEF*` translation family and a few proliferation/cytoskeleton housekeeping genes that slipped through the first pass).

In [ ]:
# 2. Load signature (file if present, else embedded pilot top-genes) + noise scrub
SIG_PATH = "out/LF_reversal_signature.json"

# Fallback used when the full signature JSON is not in THIS runtime (e.g. a fresh
# Colab session). To use the full 150/76 signature instead, upload it to
# /content/out/LF_reversal_signature.json via the Files panel and re-run this cell.
EMBEDDED = {
 "signature": "LF_activated_vs_resting_fibroblast (pilot, top genes)",
 "source": "GSE294458 (pilot, 1 HLF vs 1 NLF) - embedded top genes",
 "up": ["COL1A2","COL3A1","LUM","ASPN","COL1A1","HTRA1","MFGE8","OGN","TSC22D1","COL5A2",
        "S100A4","ARL6IP5","CD9","DCN","DKK3","CLU","MXRA8","NUPR1","SSPN","SOX5"],
 "down": ["CTSC","SERPINE1","TM4SF1","MEDAG","CD59","ANGPTL4","APOD","ACKR1","ADAMTS9",
          "IL1RL1","PFN1","CCL2","YBX1","RAN","AKAP12","EEF1B2","CXCL3","TNFRSF12A","TUBA1B","CXCL2"],
}

if os.path.exists(SIG_PATH):
    sig = json.load(open(SIG_PATH)); src = SIG_PATH
else:
    sig = EMBEDDED; src = "EMBEDDED fallback (top pilot genes)"
print("signature source:", src)
print("loaded:", sig.get("signature"), "| source:", sig.get("source"))

HB_ = {"HBB","HBA1","HBA2","HBD","HBM","HBQ1","HBZ","HBE1"}
HOUSE_ = {"MALAT1","NEAT1","XIST","ACTB","ACTG1","TMSB4X","TMSB10","B2M","GAPDH",
          "FTL","FTH1","NPM1","TPT1","NACA","MT2A","MT1X",
          "RAN","TUBA1B","TUBB","TUBB4B","PFN1","YBX1","HMGB1","HMGB2","H2AFZ",
          "STMN1","HSPA8","HSP90AA1","HSP90AB1","CALM1","CALM2"}
def _is_noise(g):
    u = g.upper()
    if u in HB_ or u in HOUSE_: return True
    if u.startswith(("RPS","RPL","MRPS","MRPL","MT-","EIF","EEF")): return True
    if re.match(r"^IG[HKL][VDJCG]", u): return True
    return False

up_genes = [g for g in sig["up"]   if not _is_noise(g)]
dn_genes = [g for g in sig["down"] if not _is_noise(g)]
print(f"UP {len(up_genes)} (was {len(sig['up'])}), DOWN {len(dn_genes)} (was {len(sig['down'])})")
print("UP  :", ", ".join(up_genes[:15]))
print("DOWN:", ", ".join(dn_genes[:15]))

## 2 — Query L1000CDS2 in reverse mode

POSTs the up/down gene sets to L1000CDS2. `aggravate:false` selects **reverse** mode (perturbagens that oppose the signature). The service returns its top matching L1000 signatures in `topMeta` (already ranked best-first).

In [ ]:
# 3. L1000CDS2 reverse query
URL = "https://maayanlab.cloud/L1000CDS2/query"
payload = {
    "data":   {"upGenes": up_genes, "dnGenes": dn_genes},
    "config": {"aggravate": False, "searchMethod": "geneSet",
               "share": True, "combination": False, "db-version": "latest"},
}
r = requests.post(URL, json=payload, headers={"content-type": "application/json"}, timeout=120)
r.raise_for_status()
res = r.json()

top = res.get("topMeta", [])
print("returned", len(top), "signatures")
if res.get("shareId"):
    print("interactive results:", "https://maayanlab.cloud/L1000CDS2/#/result/" + res["shareId"])

rows = []
for m in top:
    desc = (m.get("pert_desc") or "").strip()
    if not desc or desc in ("-666", "unannotated"):
        continue
    rows.append({"drug": desc, "score": m.get("score"),
                 "pert_id": m.get("pert_id"), "pubchem_id": m.get("pubchem_id"),
                 "cell_id": m.get("cell_id"), "sig_id": m.get("sig_id")})
raw = pd.DataFrame(rows)
print("annotated perturbagen signatures:", len(raw))
raw.head(10)

## 3 — Aggregate to compound level + apply guards

Collapse per-signature hits to one row per compound (best and mean reversal score, number of supporting signatures). Then annotate against the program's guards:

- **novelty** — compounds already named for LF in the prior-art memo are flagged `named-for-LF` (weak novelty); the specifically *clean-for-LF* classes (pirfenidone, nintedanib, metformin, senolytics, statins) are flagged `clean-for-LF (flag)`.
- **cytotoxicity caution** — MoA classes that globally suppress transcription (HDAC / proteasome / topoisomerase / HSP90 / tubulin inhibitors) can fake reversal; flagged so they are not mistaken for real hits. This is exactly the viability/matrix-preservation concern, applied at the name level here and properly at the expression level once cohorts replicate.

In [ ]:
# 4. Aggregate + guards
def norm(s): return re.sub(r"[^a-z0-9]", "", str(s).lower())

NAMED_FOR_LF = {"sirolimus","rapamycin","everolimus","temsirolimus","rolipram",
                "cyclopamine","nacetylcysteine","nacetyllcysteine","acetylcysteine",
                "2deoxydglucose","2deoxyglucose","decorin"}
CLEAN_FOR_LF = {"pirfenidone","nintedanib","metformin","dasatinib","quercetin",
                "navitoclax","fisetin","simvastatin","atorvastatin","lovastatin",
                "pravastatin","rosuvastatin","fluvastatin"}
CYTOTOX = {"vorinostat","trichostatina","panobinostat","belinostat","entinostat",
           "mocetinostat","scriptaid","hctoxin","bortezomib","mg132","doxorubicin",
           "daunorubicin","mitoxantrone","etoposide","camptothecin","topotecan",
           "paclitaxel","docetaxel","nocodazole","colchicine","geldanamycin",
           "tanespimycin","radicicol","alvespimycin","staurosporine","withaferina",
           "niclosamide","thapsigargin","digoxin","digitoxin","ouabain"}

if len(raw):
    agg = (raw.groupby("drug")
              .agg(best_score=("score","min"), mean_score=("score","mean"),
                   n_sigs=("score","size"), pubchem_id=("pubchem_id","first"))
              .reset_index())
    n = agg["drug"].map(norm)
    agg["novelty"] = "novel-for-LF"
    agg.loc[n.isin(NAMED_FOR_LF), "novelty"] = "named-for-LF (weak)"
    agg.loc[n.isin(CLEAN_FOR_LF), "novelty"] = "clean-for-LF (flag)"
    agg["cytotox_caution"] = n.isin(CYTOTOX)
    agg = agg.sort_values(["best_score","n_sigs"], ascending=[True, False]).reset_index(drop=True)
    os.makedirs("out", exist_ok=True)
    agg.to_csv("out/LF_L1000_reversal_candidates.csv", index=False)

    print("Top 20 reversers (lower score = stronger reversal):\n")
    print(agg.head(20).to_string(index=False))
    print("\nTop reversers EXCLUDING cytotoxic-caution MoAs:\n")
    print(agg[~agg["cytotox_caution"]].head(20).to_string(index=False))
    print("\nSaved out/LF_L1000_reversal_candidates.csv")
else:
    print("No annotated perturbagens returned - check the query.")

## 4 — How to read this (and what it is not)

- **Lower score = stronger predicted reversal.** Ranking is L1000CDS2's, over individual cell-line signatures collapsed to compounds.
- **Ignore the cytotoxic-caution rows for candidate purposes.** HDAC / proteasome / topoisomerase / HSP90 / tubulin inhibitors reverse almost everything by shutting transcription down; that is the opposite of a matrix-preserving anti-fibrotic. The `cytotox_caution` column exists so these do not masquerade as leads. The second printed table (cautions removed) is the more useful preview.
- **A `clean-for-LF (flag)` hit that is not cytotoxic is the most interesting outcome** — e.g. pirfenidone, nintedanib, metformin, a senolytic, or a statin surfacing as a non-cytotoxic reverser would be a genuinely novel-for-LF, locally-deliverable candidate class worth carrying into the replicated-cohort run.
- **This does not establish anything.** One 1-vs-1 pilot signature, one reversal engine, name-level guards. The real decision comes after rebuilding the signature on the Korea U (3 hypertrophic) and SMU (5v5) cohorts and requiring the same compound/mechanism to reverse across cohorts, then passing expression-level viability/matrix-preservation and local-delivery feasibility checks before any filing.